# PDF Insights Extractor
This notebook extracts text from a PDF, creates embeddings using Sentence Transformers, performs semantic search with FAISS, summarizes the document, answers questions, and logs results to JSON.

In [20]:
# Install required libraries (run once)
!pip install PyMuPDF sentence-transformers transformers faiss-cpu numpy

In [21]:
# Import libraries
import fitz  # PyMuPDF for PDF reading
import re
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline
import json

In [22]:
# Function to load and clean PDF text
def load_and_clean_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()

    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    return cleaned_text

# Example usage
pdf_path = "/content/Complete Statistics.pdf"  # Change this to your PDF file path
raw_text = load_and_clean_pdf(pdf_path)
print("Loaded text length:", len(raw_text))

Loaded text length: 39810


In [23]:
# Split text into chunks (helps with embeddings and summarization)
def chunk_text(text, chunk_size=500):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

text_chunks = chunk_text(raw_text)

In [24]:
# Generate embeddings and build FAISS index
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(text_chunks)

embeddings = np.array(embeddings).astype('float32')
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index created with", index.ntotal, "vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index created with 12 vectors


In [25]:
from transformers import BartForConditionalGeneration, BartTokenizer

# Summarization pipeline - manually load model and tokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

def generate_summary(text, max_length=150, min_length=30):
    inputs = tokenizer([text], max_length=1024, return_tensors="pt", truncation=True)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=max_length, min_length=min_length, early_stopping=True, forced_bos_token_id=0)
    return [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=False) for g in summary_ids]

# Use first chunk for summarization to avoid token limits
summary_list = generate_summary(text_chunks[0])
summary_text = summary_list[0]
print("Summary:")
print(summary_text)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
Comprehensive Guide to Statistics: Concepts, Techniques & Applications. Statistics is a vital branch of mathematics that focuses on collecting, analysing, interpreting, and presenting data. It enables informed decision-making across various industries.


In [26]:
# Question Answering pipeline
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

def answer_question(question, context):
    result = qa_pipeline(question=question, context=context)
    return result['answer'], result['score']

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# Example QA
question = "What is the main topic of the document?"
answer, confidence = answer_question(question, raw_text)

print("Question:", question)
print("Answer:", answer)
print("Confidence:", confidence)

Question: What is the main topic of the document?
Answer: Hypothesis Testing
Confidence: 0.08052780217985855


In [28]:
# Save results to JSON
results = {
    "summary": summary_text,
    "qa_logs": []
}

results["qa_logs"].append({
    "question": question,
    "answer": answer,
    "confidence": float(confidence)
})

with open("pdf_insights_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to pdf_insights_results.json")

Results saved to pdf_insights_results.json
